# Dataset Integrity Check

End-to-end verification: instantiate each PyTorch Dataset via `get_datasets()`,
iterate through **every sample**, and collect statistics on shapes, values, and errors.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import time
import traceback
from dataclasses import dataclass, field

import torch
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from radharmony.dataset import (
    BRAXDataset,
    BRAXPNGDataset,
    ChestXray14TrainDataset,
    ChestXray14TestDataset,
    ChestXray14BboxDataset,
    CheXpertTrainDataset,
    CheXpertValidDataset,
    CheXpertPlusDataset,
    CheXlocalizeDataset,
    VQARadDataset,
    ChexInstructVQADataset,
    MIMICCXRDataset,
    MIMICCXRJPGDataset,
    MIMICCXRJPGTestDataset,
    OpenICXRDataset,
    PadChestDataset,
    PadChestGRDataset,
    ReXGradientTrainDataset,
    ReXGradientValidDataset,
    ReXGradientTestDataset,
    RANZCRClipDataset,
    VinDrCXRTrainDataset,
    VinDrCXRTestDataset,
    ChestImaGenomeGoldDataset,
    ChestImaGenomeSilverDataset,
    SIIMACRPTXTrainDataset,
    SIIMCOVID19TrainDataset,
    RSNAPneumoniaKaggleTrainDataset,
    MontgomeryCXRDataset,
    ShenzhenCXRDataset,
    QaTaCOV19Dataset,
    MedSegCOVIDCTDataset,
    TAIXRay512Dataset,
    TAIXRayDataset,
    RSNAPEDetectionTrainDataset,
    RSNABoneAgeTrainDataset,
    RSNABoneAgeValDataset,
    RSNA2022CervicalSpineTrainDataset,
    RSNA2022CervicalSpineBboxDataset,
    RSNA2024LumbarSpineTrainDataset,
    RSNAAbdominalTrauma2023Dataset,
    CTRATEDataset,
    RadChestCTDataset,
)

# ── Shared integrity-check logic (also used by scripts/check_datasets.py) ──
from radharmony.integrity import (
    IntegrityReport,
    check_dataset,
    print_report,
    subsample_dataset,
    _SafeDataset,
    _collate,
)


In [3]:
# Optionally pin a GPU (dataset loading itself is CPU-bound):
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

In [4]:
# IntegrityReport is imported from radharmony.integrity (see the imports
# cell). It is defined there so this notebook and scripts/check_datasets.py
# share one source of truth and never drift.


In [5]:
# _SafeDataset, _collate, check_dataset and print_report are imported from
# radharmony.integrity — see the imports cell. Defined once there so the
# human notebook and the agent script (scripts/check_datasets.py) stay in sync.


## Configure dataset paths

Each build cell reads its paths from `configs/data_paths.json` with `data_paths("<registry key>")`.
To use another file, set `RADHARMONY_DATA_PATHS` before you start Jupyter.
A dataset with no entry in your file is skipped with a hint.

In [ ]:
# ── Dataset paths ──────────────────────────────────────────────────────────
# Paths come from configs/data_paths.json (read with scripts/dataset_paths.py).
import sys

sys.path.insert(0, "../../scripts")
from dataset_paths import MISSING, paths


def data_paths(key, variant=None, only=None):
    """Return the path arguments of a registry key, or {MISSING: hint} when your file has no entry."""
    return paths(key, variant, only=only, missing_ok=True)


# Scratch dir for datasets that rasterize/decode masks to PNG on the fly.
MASK_OUT = os.path.expanduser("~/.cache/radharmony/verify_masks")
os.makedirs(MASK_OUT, exist_ok=True)

## Verification driver

`build(DatasetClass, **kwargs)` does the whole loop for one dataset:

1. **instantiate** the dataset with the given (all supported) arguments — this
   runs the harmonizer end-to-end;
2. **sub-sample** the harmonized table to `MAX_SAMPLES` rows for a fast pass
   (set `MAX_SAMPLES = None` for a full integrity sweep of every sample);
3. **iterate** every (sub-sampled) sample through a `DataLoader` and collect
   shape / range / NaN / error statistics (`check_dataset`);
4. **preview** — loop a few samples, **print the decoded labels / report /
   bbox**, and **visualize** the image (2-D directly, 3-D as an axial mean
   projection; masks and bboxes overlaid when present).

Per-sample and per-dataset errors are captured, not raised, so one broken
dataset never aborts the run — it just shows up as `ERROR`/`WARN` in its report
and in the final summary table.

In [ ]:
# ── Verification driver ───────────────────────────────────────────────────
MAX_SAMPLES = 60    # rows to sub-sample per dataset (None = every sample)
N_PREVIEW   = 4     # samples to print + visualize per dataset


def _subsample(ds):
    """Pin a small random subset of the harmonized table for a fast pass."""
    n0, n1 = subsample_dataset(ds, MAX_SAMPLES)
    if n1 < n0:
        print(f"  sub-sampled {n0:,} -> {n1} rows (set MAX_SAMPLES=None for all)")


def preview(r, n=N_PREVIEW):
    """Loop a few samples through a DataLoader: print labels/report/bbox, show figures."""
    import matplotlib.pyplot as plt

    ds_obj = r.ds_obj
    if ds_obj is None or r.success == 0:
        print("  (no sample loaded - nothing to preview)")
        return
    try:
        dataset = ds_obj.get_datasets()
    except Exception as e:
        print(f"  [warn] could not build dataset for preview: {type(e).__name__}: {e}")
        return
    lcols = sorted(ds_obj.LABEL_COLS) if getattr(ds_obj, "LABEL_COLS", None) else []
    rcols = sorted(getattr(ds_obj, "REG_COLS", []) or [])

    loader = DataLoader(
        _SafeDataset(dataset), batch_size=1, num_workers=0, collate_fn=_collate
    )
    print(f"  -- looping {min(n, len(dataset))} sample(s) through a DataLoader --")
    shown = 0
    for batch in loader:
        idx = batch["_idx"][0]
        if "img" not in batch:
            print(f"    sample[{idx}] FAILED: {batch['_err'][0]}")
            continue
        img = batch["img"][0]
        print(f"    sample[{idx}]  img shape={tuple(img.shape)} dtype={img.dtype}")
        if "cls" in batch:
            v = batch["cls"][0]
            v = v.float().tolist() if hasattr(v, "float") else [float(x) for x in v]
            pos = [lcols[j] for j, x in enumerate(v)
                   if j < len(lcols) and not np.isnan(x) and x > 0.5]
            print(f"        cls[{len(v)}] positive: {pos if pos else '(none)'}")
        if "reg" in batch:
            v = batch["reg"][0]
            v = v.float().tolist() if hasattr(v, "float") else list(v)
            print(f"        reg {rcols} = {v}")
        if "bbox" in batch:
            labs = batch.get("bbox_labels", [None])[0]
            print(f"        bbox: {len(batch['bbox'][0])} box(es)  labels={labs}")
        if "report" in batch:
            print(f"        report: {str(batch['report'][0])[:150].strip()} ...")
        if "previous_img" in batch:
            print(f"        previous_img shape={tuple(batch['previous_img'][0].shape)}")
        if "question" in batch:
            print(f"        question: {str(batch['question'][0]).strip()}")
        if "answer" in batch:
            print(f"        answer: {str(batch['answer'][0]).strip()}")
        if "question_type" in batch:
            print(f"        question_type: {batch['question_type'][0]}")
        shown += 1
        if shown >= n:
            break

    try:
        for fig in ds_obj.visualize_samples(dataset, n=n, random_state=0):
            plt.show()
            plt.close(fig)
    except Exception as e:
        print(f"  [warn] visualize_samples failed: {type(e).__name__}: {e}")


def build(ds_cls, *, num_workers=4, **kwargs):
    """Instantiate -> sub-sample -> integrity-check -> preview one dataset."""
    if MISSING in kwargs:
        print(f"# {ds_cls.__name__}: skipped. {kwargs[MISSING]}")
        return None
    kwargs.setdefault("cache_dir", None)
    shown = {k: v for k, v in kwargs.items() if k != "cache_dir"}
    print("\n" + "#" * 72)
    print(f"# {ds_cls.__name__}")
    for k, v in shown.items():
        print(f"#   {k} = {v}")
    print("#" * 72)
    try:
        ds = ds_cls(**kwargs)
    except Exception as e:
        traceback.print_exc()
        r = IntegrityReport(dataset=ds_cls.__name__, status="ERROR",
                            error_message=f"__init__ failed: {e}")
        reports.append((r, []))
        return r
    try:
        _subsample(ds)
    except Exception as e:
        print(f"  [warn] could not sub-sample: {type(e).__name__}: {e}")
    label_cols = getattr(ds, "LABEL_COLS", []) or []
    r = check_dataset(ds, num_workers=num_workers)
    print_report(r, label_cols)
    preview(r)
    reports.append((r, label_cols))
    return r

## Run verification

Each cell builds one dataset, checks integrity over a sub-sample, prints labels, and shows previews. Run top-to-bottom or cherry-pick the datasets you have.

In [8]:
reports = []  # (IntegrityReport, label_cols) per dataset

### VQA

In [ ]:
# VQA-RAD — 315 MedPix images, 2,248 questions (no patient/study concept;
# patient_id/study_id are both derived from the image filename).
build(VQARadDataset, **data_paths("vqa_rad"), output_question_type=True)

In [ ]:
# CheXinstruct VQA (MIMIC-CXR-VQA + MIMIC-Diff-VQA) — Open-Ended VQA,
# Close-Ended VQA, and Difference VQA tasks from Stanford's CheXinstruct,
# translated from a locally-compiled JSON (see docs/wiki/reference/datasets/vqa-text/chexinstruct_vqa.md).
build(
    ChexInstructVQADataset,
    **data_paths("chexinstruct_vqa"),
    output_struct=True,
    output_question_type=True,
)


### Chest X-ray (2-D)

In [ ]:
# # CheXpert — train (14 pathologies)
# build(CheXpertTrainDataset, **data_paths("chexpert_train"),
#       output_cls=True, drop_uncertain=False)

In [ ]:
# # CheXpert — valid
# build(CheXpertValidDataset, **data_paths("chexpert_valid"),
#       output_cls=True)

In [ ]:
# # CheXpert-Plus (DICOM) — cls + de-identified reports
# build(CheXpertPlusDataset, **data_paths("chexpert_plus"),
#       output_cls=True, output_report=True, drop_uncertain=False)

In [ ]:
# CheXlocalize (CheXpert "test" split) — cls (14 pathologies). Enable mask via the next cell.
build(CheXlocalizeDataset, **data_paths("chexlocalize", only=("base_image_dir", "csv_path")),
      output_cls=True)

In [ ]:
# CheXlocalize — cls + unioned segmentation mask (per-pathology COCO-RLE
# decoded and OR'd into one binary mask per image; 499/668 images have one).
build(CheXlocalizeDataset, **data_paths("chexlocalize"),
      mask_output_dir=os.path.join(MASK_OUT, "chexlocalize"),
      output_cls=True, output_mask=True)

In [ ]:
# # MIMIC-CXR (DICOM) — cls + reports (metadata/labels may live apart from the image tree)
# build(MIMICCXRDataset, **data_paths("mimic_cxr"),
#       output_cls=True, output_report=True, drop_uncertain=False)

In [ ]:
# MIMIC-CXR-JPG — cls (JPG release ships no reports)
build(MIMICCXRJPGDataset, **data_paths("mimic_cxr_jpg"), output_cls=True)

In [ ]:
# MIMIC-CXR-JPG — labeled test set
build(MIMICCXRJPGTestDataset, **data_paths("mimic_cxr_jpg_test"), output_cls=True)

In [ ]:
# ChestX-ray14 — train (15 classes)
build(ChestXray14TrainDataset, **data_paths("chestxray14_train"), output_cls=True)

In [ ]:
# ChestX-ray14 — test
build(ChestXray14TestDataset, **data_paths("chestxray14_test"), output_cls=True)

In [ ]:
# ChestX-ray14 — bbox subset (cls + bbox)
build(ChestXray14BboxDataset, **data_paths("chestxray14_bbox"), output_cls=True, output_bbox=True)

In [ ]:
# PadChest — 193 labels + Spanish reports
build(PadChestDataset, **data_paths("padchest"),
      output_cls=True, output_report=True)

In [ ]:
# PadChest-GR — 154 labels, sentence-grounded boxes, EN/ES reports
# Set PADCHEST_GR_DIR to use another image folder.
build(PadChestGRDataset, **data_paths("padchest_gr"),
      output_cls=True, output_report=True, output_bbox=True)

In [ ]:
# BRAX (DICOM) — cls (uncertain -1 -> 0)
build(BRAXDataset, **data_paths("brax"),
      uncertain_strategy="u_zeros", output_cls=True)

In [ ]:
# BRAX (PNG) — cls
build(BRAXPNGDataset, **data_paths("brax_png"),
      uncertain_strategy="u_zeros", output_cls=True)

In [ ]:
# OpenI IU CXR — cls (CheXpert-compat) + reports
build(OpenICXRDataset, **data_paths("openi_cxr"), output_cls=True, output_report=True)

In [ ]:
# ReXGradient-160K — valid (report only, no labels)
build(ReXGradientValidDataset, **data_paths("rexgradient_valid"), output_report=True)

In [ ]:
# ReXGradient-160K — test
build(ReXGradientTestDataset, **data_paths("rexgradient_test"), output_report=True)

In [ ]:
# ReXGradient-160K — train (large; sub-sampled after harmonize)
build(ReXGradientTrainDataset, **data_paths("rexgradient_train"), output_report=True)

In [ ]:
# VinDr-CXR — train (cls + bbox)
build(VinDrCXRTrainDataset, **data_paths("vindr_cxr_train"), output_cls=True, output_bbox=True)

In [ ]:
# VinDr-CXR — test
build(VinDrCXRTestDataset, **data_paths("vindr_cxr_test"), output_cls=True, output_bbox=True)

In [ ]:
# Chest ImaGenome - gold (1,000 verified images; per-box anatomy + findings)
# Anatomy in bbox_labels; per-box findings in the harmonized bbox_findings column.
build(ChestImaGenomeGoldDataset, **data_paths("chest_imagenome_gold"), output_bbox=True)

In [ ]:
# Chest ImaGenome - silver (full scene-graph release; 'valid' split for a quick pass).
# NOTE: harmonize reads one DICOM header per image, so a full split is slow.
build(ChestImaGenomeSilverDataset, **data_paths("chest_imagenome_silver"),
      split="valid", output_bbox=True)

In [ ]:
# SIIM-ACR Pneumothorax — cls + mask (RLE decoded to PNG under MASK_OUT)
build(SIIMACRPTXTrainDataset, **data_paths("siim_acr_ptx_train"),
      mask_output_dir=os.path.join(MASK_OUT, "siim_acr"), output_cls=True, output_mask=True)

In [ ]:
# SIIM COVID-19 — cls (4 study classes) + bbox
build(SIIMCOVID19TrainDataset, **data_paths("siim_covid19_train"), output_cls=True, output_bbox=True)

In [ ]:
# RSNA Pneumonia (Kaggle) — cls (3 classes) + bbox
build(RSNAPneumoniaKaggleTrainDataset, **data_paths("rsna_pneumonia_kaggle_train"),
      output_cls=True, output_bbox=True)

In [ ]:
# RANZCR CLiP — cls (11 line/catheter labels). Enable mask via the next cell.
build(RANZCRClipDataset, **data_paths("ranzcr_clip", only=("base_image_dir", "csv_path")),
      include_test_split=False, output_cls=True)

In [ ]:
# RANZCR CLiP — cls + rasterized polyline masks (slower: masks are
# generated at harmonize time). Restrict to annotated rows so every
# sample has a real mask.
build(RANZCRClipDataset, **data_paths("ranzcr_clip"),
      mask_output_dir=os.path.join(MASK_OUT, "ranzcr"),
      include_test_split=False, output_cls=True, output_mask=True)

In [ ]:
# TAIX-Ray 512px — cls, ordinal severity grades (0-4)
build(TAIXRay512Dataset, **data_paths("taix_ray_512"),
      label_mode="ordinal", output_cls=True)

In [ ]:
# Montgomery County CXR — TB cls + lung mask + report
build(MontgomeryCXRDataset, **data_paths("montgomery_cxr"),
      mask_output_dir=os.path.join(MASK_OUT, "montgomery"),
      output_cls=True, output_mask=True, output_report=True)

In [ ]:
# Shenzhen Hospital CXR — TB cls + mask + report
build(ShenzhenCXRDataset, **data_paths("shenzhen_cxr"),
      mask_output_dir=os.path.join(MASK_OUT, "shenzhen"),
      output_cls=True, output_mask=True, output_report=True)

In [ ]:
# QaTa-COV19 v2 (train) — COVID cls (constant 1) + infection-region mask
# One class serves both splits; the Images/ directory picks the split.
# No mask_output_dir — QaTa ships ready-made binary mask PNGs.
build(QaTaCOV19Dataset, **data_paths("qata_cov19", "train"),
      output_cls=True, output_mask=True)

In [ ]:
# QaTa-COV19 v2 (test) — COVID cls (constant 1) + infection-region mask
build(QaTaCOV19Dataset, **data_paths("qata_cov19", "test"),
      output_cls=True, output_mask=True)

In [ ]:
# MedSeg COVID-19 CT — dataset1 (100 annotated slices), real multi-label lesion flags
build(MedSegCOVIDCTDataset, **data_paths("medseg_covid_ct", "dataset1"), subset="dataset1",
      output_cls=True, output_mask=True)

In [ ]:
# MedSeg COVID-19 CT — dataset2 (9 volumes, 829 slices; ~55% are lesion-free)
build(MedSegCOVIDCTDataset, **data_paths("medseg_covid_ct", "dataset2"), subset="dataset2",
      output_cls=True, output_mask=True)

### CT / MRI (3-D)

In [ ]:
# RSNA PE Detection (3-D CTPA series) — cls
build(RSNAPEDetectionTrainDataset, **data_paths("rsna_pe_detection_train"),
      output_cls=True, num_workers=2)

In [ ]:
# RSNA Pediatric Bone Age (2-D) — regression (age_months)
build(RSNABoneAgeTrainDataset, **data_paths("rsna_bone_age_train"), output_reg=True)

In [ ]:
# RSNA 2022 Cervical Spine (3-D) — cls + NIfTI seg mask
build(RSNA2022CervicalSpineTrainDataset, **data_paths("rsna_2022_cervical_spine_train"),
      output_cls=True, output_mask=True, num_workers=2)

In [ ]:
# RSNA 2022 Cervical Spine — bbox variant (cls + 3-D bbox)
build(RSNA2022CervicalSpineBboxDataset, **data_paths("rsna_2022_cervical_spine_bbox"),
      output_cls=True, output_bbox=True, num_workers=2)

In [ ]:
# RSNA 2024 Lumbar Spine (3-D MRI) — cls (75 cols) + bbox
build(RSNA2024LumbarSpineTrainDataset, **data_paths("rsna_2024_lumbar_spine_train"),
      output_cls=True, output_bbox=True, num_workers=2)

In [ ]:
# RSNA 2023 Abdominal Trauma (3-D) — cls (14 injury labels)
build(RSNAAbdominalTrauma2023Dataset, **data_paths("rsna_abdominal_trauma_2023"),
      output_cls=True, num_workers=2)

In [ ]:
# CT-RATE (3-D NIfTI) — cls (18 findings)
build(CTRATEDataset, **data_paths("ct_rate"), output_cls=True, num_workers=2)

In [ ]:
# RAD-ChestCT (3-D .npz) — cls (83/84 findings) + bbox
build(RadChestCTDataset, **data_paths("radchestct"),
      output_cls=True, output_bbox=True, num_workers=2)

## Summary table

In [43]:
summary_df = pd.DataFrame([r.summary_dict() for r, _ in reports])
summary_df

,dataset,status,samples,success,failed,img_range,img_nan,img_shapes,time_s
0,MIMICCXRJPGDataset,OK,60,60,0,"[-1.000, 1.000]",0,"(1, 224, 224): 60",3.2
1,MIMICCXRJPGTestDataset,OK,60,60,0,"[-1.000, 1.000]",0,"(1, 224, 224): 60",1.9
2,ChestXray14TrainDataset,OK,60,60,0,"[-1.000, 1.000]",0,"(1, 224, 224): 59, (4, 224, 224): 1",0.8
3,ChestXray14TestDataset,OK,60,60,0,"[-1.000, 1.000]",0,"(1, 224, 224): 60",0.7
4,ChestXray14BboxDataset,OK,60,60,0,"[-1.000, 1.000]",0,"(1, 224, 224): 60",0.8
5,PadChestDataset,OK,60,60,0,"[-1.000, 1.000]",0,"(1, 224, 224): 60",2.1
6,BRAXDataset,OK,60,60,0,"[-1.000, 1.000]",0,"(1, 224, 224): 60",2.1
7,BRAXPNGDataset,OK,60,60,0,"[-1.000, 1.000]",0,"(1, 224, 224): 60",1.0
8,OpenICXRDataset,ERROR,0,0,0,-,0,,0.0
9,ReXGradientValidDataset,OK,60,60,0,"[-1.000, 1.000]",0,"(1, 224, 224): 60",1.3


## Label distribution overview

In [44]:
for r, lcols in reports:
    if r.success == 0 or not r.has_cls:
        continue
    rows = []
    for idx in range(r.cls_length):
        name = lcols[idx] if idx < len(lcols) else f"[{idx}]"
        pos = r.cls_pos_counts.get(idx, 0)
        neg = r.success - pos
        rows.append(
            {
                "label": name,
                "positive": pos,
                "negative": neg,
                "pos_rate": f"{pos / r.success * 100:.1f}%" if r.success else "-",
            }
        )
    print(f"\n--- {r.dataset} ---")
    display(pd.DataFrame(rows))


--- MIMICCXRJPGDataset ---


,label,positive,negative,pos_rate
0,atelectasis,8,52,13.3%
1,cardiomegaly,16,44,26.7%
2,consolidation,1,59,1.7%
3,edema,5,55,8.3%
4,enlarged_cardiomediastinum,0,60,0.0%
5,fracture,3,57,5.0%
6,lung_lesion,1,59,1.7%
7,lung_opacity,7,53,11.7%
8,no_finding,28,32,46.7%
9,pleural_effusion,10,50,16.7%



--- MIMICCXRJPGTestDataset ---


,label,positive,negative,pos_rate
0,atelectasis,23,37,38.3%
1,cardiomegaly,9,51,15.0%
2,consolidation,1,59,1.7%
3,edema,6,54,10.0%
4,enlarged_cardiomediastinum,3,57,5.0%
5,fracture,4,56,6.7%
6,lung_lesion,4,56,6.7%
7,lung_opacity,15,45,25.0%
8,no_finding,5,55,8.3%
9,pleural_effusion,22,38,36.7%



--- ChestXray14TrainDataset ---


,label,positive,negative,pos_rate
0,atelectasis,6,54,10.0%
1,cardiomegaly,2,58,3.3%
2,consolidation,2,58,3.3%
3,edema,1,59,1.7%
4,effusion,9,51,15.0%
5,emphysema,0,60,0.0%
6,fibrosis,0,60,0.0%
7,hernia,0,60,0.0%
8,infiltration,10,50,16.7%
9,mass,6,54,10.0%


... (36 outputs left out)


--- CTRATEDataset ---


,label,positive,negative,pos_rate
0,arterial_wall_calcification,15,45,25.0%
1,atelectasis,16,44,26.7%
2,bronchiectasis,4,56,6.7%
3,cardiomegaly,6,54,10.0%
4,consolidation,12,48,20.0%
5,coronary_artery_wall_calcification,12,48,20.0%
6,emphysema,9,51,15.0%
7,hiatal_hernia,7,53,11.7%
8,interlobular_septal_thickening,5,55,8.3%
9,lung_nodule,28,32,46.7%



--- RadChestCTDataset ---


,label,positive,negative,pos_rate
0,air_trapping,2,58,3.3%
1,airspace_disease,8,52,13.3%
2,aneurysm,1,59,1.7%
3,arthritis,13,47,21.7%
4,aspiration,1,59,1.7%
...,...,...,...,...
79,suture,1,59,1.7%
80,tracheal_tube,0,60,0.0%
81,transplant,3,57,5.0%
82,tree_in_bud,2,58,3.3%
